# AgroBot — Rover Path Detection CNN

**Project:** Smart Agriculture System for Nepal (FYP)  
**Author:** Prashant Rijal  

## Overview
Trains a **U-Net** (PyTorch) to perform pixel-wise segmentation of navigable
paths in rover camera images.  The Flask API (`POST /api/rover/path`) loads
the saved weights and returns:
- `direction`  — LEFT / STRAIGHT / RIGHT / STOP  
- `coverage`   — % of frame that is navigable path  
- `mask_base64`— PNG overlay for the dashboard  

### Dataset
Synthetic images generated with NumPy/Pillow: green crop-row backgrounds +
a curved dirt-path strip of varying width, position and brightness.

### Architecture
Lightweight U-Net — ~490 K parameters, fast CPU inference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

np.random.seed(42)
torch.manual_seed(42)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE   = 128
N_SAMPLES  = 3000
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-3

print(f'PyTorch  {torch.__version__}  |  device: {DEVICE}')

## 1. Synthetic Dataset

In [ ]:
def _crop_texture(img, size):
    img[:, :, 1] = np.random.randint(55, 110, (size, size), dtype=np.uint8)  # green
    img[:, :, 0] = np.random.randint(15,  50, (size, size), dtype=np.uint8)  # red
    img[:, :, 2] = np.random.randint(10,  40, (size, size), dtype=np.uint8)  # blue
    # Vertical crop rows
    for x in range(0, size, np.random.randint(8, 18)):
        w = np.random.randint(1, 4)
        x1, x2 = max(0, x - w), min(size, x + w)
        img[:, x1:x2, 1] = np.clip(img[:, x1:x2, 1].astype(int) - 20, 0, 255).astype(np.uint8)


def _draw_path(img, mask, size):
    path_width = np.random.randint(22, 55)
    cx = np.random.randint(path_width // 2 + 5, size - path_width // 2 - 5)
    for row in range(size):
        cx = int(np.clip(cx + np.random.normal(0, 1.2), path_width // 2, size - path_width // 2))
        l, r = max(0, cx - path_width // 2), min(size, cx + path_width // 2)
        w = r - l
        img[row, l:r, 0] = np.random.randint(100, 155, w, dtype=np.uint8)
        img[row, l:r, 1] = np.random.randint(80,  120, w, dtype=np.uint8)
        img[row, l:r, 2] = np.random.randint(50,   90, w, dtype=np.uint8)
        mask[row, l:r]   = 1


def generate_sample(size=IMG_SIZE):
    img  = np.zeros((size, size, 3), dtype=np.uint8)
    mask = np.zeros((size, size),    dtype=np.uint8)
    _crop_texture(img, size)
    _draw_path(img, mask, size)
    # Random brightness / contrast
    alpha = np.random.uniform(0.7, 1.3)
    beta  = np.random.randint(-20, 20)
    img   = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return img, mask


# Preview
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for col in range(5):
    im, mk = generate_sample()
    axes[0, col].imshow(im);  axes[0, col].set_title('Image', fontsize=9); axes[0, col].axis('off')
    axes[1, col].imshow(mk, cmap='gray'); axes[1, col].set_title('Mask', fontsize=9); axes[1, col].axis('off')
plt.suptitle('Synthetic Field Path Samples', fontweight='bold')
plt.tight_layout()
os.makedirs('../figures and Diagrams', exist_ok=True)
plt.savefig('../figures and Diagrams/path_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Build Dataset

In [ ]:
class PathDataset(Dataset):
    def __init__(self, n):
        self.images = np.zeros((n, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        self.masks  = np.zeros((n, 1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        for i in range(n):
            img, mask = generate_sample()
            self.images[i] = img.astype(np.float32).transpose(2, 0, 1) / 255.0
            self.masks[i, 0] = mask.astype(np.float32)
    def __len__(self):  return len(self.images)
    def __getitem__(self, idx):
        return torch.from_numpy(self.images[idx]), torch.from_numpy(self.masks[idx])


print(f'Generating {N_SAMPLES} samples ...', end=' ', flush=True)
full_ds = PathDataset(N_SAMPLES)
print('done.')

val_size   = int(N_SAMPLES * 0.15)
train_size = N_SAMPLES - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {train_size}  |  Val: {val_size}')

## 3. U-Net Model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, filters=(16, 32, 64, 128)):
        super().__init__()
        f = filters
        self.enc1   = ConvBlock(3,    f[0])
        self.enc2   = ConvBlock(f[0], f[1])
        self.enc3   = ConvBlock(f[1], f[2])
        self.bridge = nn.Sequential(ConvBlock(f[2], f[3]), nn.Dropout2d(0.3))
        self.up3    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec3   = ConvBlock(f[3] + f[2], f[2])
        self.up2    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2   = ConvBlock(f[2] + f[1], f[1])
        self.up1    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1   = ConvBlock(f[1] + f[0], f[0])
        self.pool   = nn.MaxPool2d(2)
        self.out    = nn.Conv2d(f[0], 1, 1)

    def forward(self, x):
        c1 = self.enc1(x);  p1 = self.pool(c1)
        c2 = self.enc2(p1); p2 = self.pool(c2)
        c3 = self.enc3(p2); p3 = self.pool(c3)
        cb = self.bridge(p3)
        d3 = self.dec3(torch.cat([self.up3(cb), c3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), c2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), c1], dim=1))
        return torch.sigmoid(self.out(d1))


model     = UNet().to(DEVICE)
total_par = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_par:,}')

## 4. Loss Functions and Training Loop

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    p = pred.view(-1); t = target.view(-1)
    inter = (p * t).sum()
    return 1 - (2 * inter + smooth) / (p.sum() + t.sum() + smooth)

def bce_dice(pred, target):
    return nn.functional.binary_cross_entropy(pred, target) + dice_loss(pred, target)

def iou_score(pred, target, threshold=0.5):
    p = (pred > threshold).float(); t = target
    inter = (p * t).sum(); union = p.sum() + t.sum() - inter
    return (inter / (union + 1e-6)).item()


optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                  factor=0.5, patience=3)

train_losses, val_ious = [], []
best_iou, patience_cnt = 0.0, 0
PATIENCE = 7

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    t_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = bce_dice(preds, masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ---- Validate ----
    model.eval()
    v_iou = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            v_iou += iou_score(model(imgs), masks)
    v_iou /= len(val_loader)

    train_losses.append(t_loss)
    val_ious.append(v_iou)
    scheduler.step(v_iou)

    if v_iou > best_iou:
        best_iou = v_iou
        torch.save(model.state_dict(), '/tmp/best_path_model.pt')
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:3d}/{EPOCHS}  loss={t_loss:.4f}  val_IoU={v_iou*100:.1f}%')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest Validation IoU: {best_iou*100:.1f}%')

## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss');  ax1.set_title('BCE + Dice Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot([v*100 for v in val_ious], color='#10b981', label='Val IoU %')
ax2.set_title('Validation IoU', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures and Diagrams/path_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Prediction Samples

In [ ]:
def mask_to_direction(mask_np, threshold=0.5):
    binary    = (mask_np > threshold).astype(np.uint8)
    h, w      = binary.shape
    roi       = binary[int(h * 0.4):, :]
    path_cols = np.where(roi.sum(axis=0) > roi.shape[0] * 0.12)[0]
    if len(path_cols) == 0: return 'STOP'
    offset = (path_cols.mean() - w / 2) / (w / 2)
    if   offset < -0.2: return 'LEFT'
    elif offset >  0.2: return 'RIGHT'
    else:               return 'STRAIGHT'


# Load best weights for visualization
model.load_state_dict(torch.load('/tmp/best_path_model.pt', map_location=DEVICE))
model.eval()

n_show = 6
sample_imgs, sample_masks = next(iter(DataLoader(val_ds, batch_size=n_show, shuffle=True)))
with torch.no_grad():
    preds = model(sample_imgs.to(DEVICE)).cpu().numpy()

fig, axes = plt.subplots(3, n_show, figsize=(16, 7))
for col in range(n_show):
    direction = mask_to_direction(preds[col, 0])
    axes[0, col].imshow(sample_imgs[col].permute(1, 2, 0).numpy())
    axes[0, col].set_title('Input', fontsize=8); axes[0, col].axis('off')
    axes[1, col].imshow(sample_masks[col, 0].numpy(), cmap='gray')
    axes[1, col].set_title('GT Mask', fontsize=8); axes[1, col].axis('off')
    color = '#10b981' if direction == 'STRAIGHT' else '#f59e0b'
    axes[2, col].imshow(preds[col, 0], cmap='YlGn', vmin=0, vmax=1)
    axes[2, col].set_title(f'→ {direction}', fontsize=8, color=color)
    axes[2, col].axis('off')

plt.suptitle('Path Detection — Predictions vs Ground Truth', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures and Diagrams/path_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Direction Accuracy

In [ ]:
from collections import Counter

all_imgs  = torch.from_numpy(full_ds.images[train_size:]).to(DEVICE)
all_masks = full_ds.masks[train_size:, 0]

with torch.no_grad():
    all_preds = model(all_imgs).cpu().numpy()[:, 0]

pred_dirs = [mask_to_direction(p) for p in all_preds]
true_dirs = [mask_to_direction(m) for m in all_masks]

direction_acc = sum(p == t for p, t in zip(pred_dirs, true_dirs)) / len(true_dirs)
print(f'Direction Accuracy : {direction_acc*100:.1f}%')
print('True  dist:', Counter(true_dirs))
print('Pred  dist:', Counter(pred_dirs))

## 8. Save Model

In [ ]:
import shutil

model_dir = '../website/models'
os.makedirs(model_dir, exist_ok=True)

dst = os.path.join(model_dir, 'path_model.pt')
shutil.copy('/tmp/best_path_model.pt', dst)

size_kb = os.path.getsize(dst) / 1024
print(f'Model saved  → {dst}  ({size_kb:.0f} KB)')
print(f'Best IoU     : {best_iou*100:.1f}%')
print(f'Direction Acc: {direction_acc*100:.1f}%')
print(f'Resolution   : {IMG_SIZE}×{IMG_SIZE} RGB  →  {IMG_SIZE}×{IMG_SIZE} binary mask')
print('\nTo use in Flask: POST image to /api/rover/path')